In [1]:
"""HOG round-fps-v1: keyframe cách nhau round(fps) frame, mỗi vector có 8100 chiều."""

import csv
import json
from pathlib import Path

import cv2
import numpy as np
from skimage.feature import hog


def extract_video_keyframes(video_path: str | Path) -> list[dict]:
    """Lấy frame 0, frame_interval, 2 * frame_interval... như thuật toán đã chọn."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        raise ValueError(f"Không mở được video: {video_path}")

    keyframes = []
    try:
        fps = cap.get(cv2.CAP_PROP_FPS)
        if not np.isfinite(fps) or fps <= 0:
            fps = 25

        frame_interval = max(1, int(round(fps)))
        frame_idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % frame_interval == 0:
                keyframes.append({
                    "frame_idx": frame_idx,
                    "timestamp": frame_idx / fps,
                    "frame": frame,
                    "fps": fps,
                    "frame_interval": frame_interval,
                })
            frame_idx += 1
    finally:
        cap.release()

    if not keyframes:
        raise ValueError(f"Video không có frame đọc được: {video_path}")
    return keyframes


def extract_hog_keyframe(frame: np.ndarray) -> np.ndarray:
    """Resize toàn frame về 128x128, chuyển xám và trích HOG L2-Hys."""
    resized = cv2.resize(frame, (128, 128), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    return hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=False,
        feature_vector=True,
        channel_axis=None,
    ).astype(np.float32)


def extract_video_hog(video_path: str | Path) -> list[dict]:
    """Trả về một bản ghi cho mỗi keyframe, giữ riêng từng vector HOG."""
    video_path = Path(video_path).resolve()
    return [
        {
            "filename": video_path.name,
            "filepath": str(video_path),
            "label": video_path.parent.name,
            "keyframe_idx": keyframe_idx,
            "frame_idx": keyframe["frame_idx"],
            "timestamp_sec": keyframe["timestamp"],
            "fps": keyframe["fps"],
            "frame_interval": keyframe["frame_interval"],
            "feature_dim": 8100,
            "vector": extract_hog_keyframe(keyframe["frame"]),
        }
        for keyframe_idx, keyframe in enumerate(extract_video_keyframes(video_path))
    ]


def write_hog_csv(records, csv_path: str | Path) -> None:
    """Ghi mỗi keyframe thành một dòng CSV; vector là mảng số theo định dạng JSON."""
    fields = [
        "filename", "filepath", "label", "keyframe_idx", "frame_idx",
        "timestamp_sec", "fps", "frame_interval", "feature_dim", "vector",
    ]
    with Path(csv_path).open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        for record in records:
            writer.writerow({
                **record,
                "vector": json.dumps(
                    record["vector"].tolist(), separators=(",", ":"), allow_nan=False
                ),
            })


In [2]:
from d23_lop4_nhom16.dataset import VideoDogDataset 
from d23_lop4_nhom16.config import RAW_DATASET_DIR

In [3]:
dataset = VideoDogDataset(RAW_DATASET_DIR) 

In [4]:

all_features = [] 
    
for video_path in dataset.video_paths: 
    video_features = extract_video_hog(video_path) 
    all_features.extend(video_features) 
    print(
        f"Video: {video_path} | "
        f"Total extracted frames: {len(video_features)}"
    )


Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_1.mp4 | Total extracted frames: 16
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_20.mp4 | Total extracted frames: 6
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_66.mp4 | Total extracted frames: 14
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_54.mp4 | Total extracted frames: 19
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_59.mp4 | Total extracted frames: 12
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_50.mp4 | Total extracted frames: 9
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_5.mp4 | Total extracted frames: 16
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_55.mp4 | Total extracted frames: 12
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_13.mp4 | Total extracted frames: 15
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/Eating/dog_eat_

[h264 @ 0x3c567000] sps_id 1 out of range
[h264 @ 0x3c567000] sps_id 1 out of range
[h264 @ 0x3c567000] sps_id 1 out of range
[h264 @ 0x3c567000] sps_id 1 out of range
[h264 @ 0x3bb83a80] sps_id 1 out of range
[h264 @ 0x3bb83a80] sps_id 1 out of range


Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_14.mp4 | Total extracted frames: 6
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_31.mp4 | Total extracted frames: 16
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_71.mp4 | Total extracted frames: 32
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_20.mp4 | Total extracted frames: 6


[h264 @ 0x3b116dc0] sps_id 1 out of range
[h264 @ 0x3b116dc0] sps_id 1 out of range
[h264 @ 0x3b116dc0] sps_id 1 out of range
[h264 @ 0x3b116dc0] sps_id 1 out of range
[h264 @ 0x3b9e2a40] sps_id 1 out of range
[h264 @ 0x3b9e2a40] sps_id 1 out of range


Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/walking/dog_walking_50.mp4 | Total extracted frames: 7


[h264 @ 0x3bf47b80] Clipping illegal num_reorder_frames 32
[h264 @ 0x3bf47b80] sps_id 1 out of range
[h264 @ 0x3bf47b80] sps_id 1 out of range
[h264 @ 0x3bf47b80] Clipping illegal num_reorder_frames 32
[h264 @ 0x3bf47b80] sps_id 1 out of range
[h264 @ 0x3bf47b80] sps_id 1 out of range
[h264 @ 0x3bf59940] Clipping illegal num_reorder_frames 32
[h264 @ 0x3bf59940] sps_id 1 out of range
[h264 @ 0x3bf59940] sps_id 1 out of range


... (71 outputs left out)

[h264 @ 0x3c123d80] sps_id 1 out of range
[h264 @ 0x3c123d80] sps_id 1 out of range
[h264 @ 0x3c123d80] sps_id 1 out of range
[h264 @ 0x3c123d80] sps_id 1 out of range
[h264 @ 0x3f9d5cc0] sps_id 1 out of range
[h264 @ 0x3f9d5cc0] sps_id 1 out of range


Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_62.mp4 | Total extracted frames: 2
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_90.mp4 | Total extracted frames: 16
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_21.mp4 | Total extracted frames: 13
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_52.mp4 | Total extracted frames: 18
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_14.mp4 | Total extracted frames: 4
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_58.mp4 | Total extracted frames: 17
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_42.mp4 | Total extracted frames: 24
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_67.mp4 | Total extracted frames: 17


[h264 @ 0x3b17d140] sps_id 1 out of range
[h264 @ 0x3b17d140] sps_id 1 out of range
[h264 @ 0x3b17d140] sps_id 1 out of range
[h264 @ 0x3b17d140] sps_id 1 out of range
[h264 @ 0x41d13dc0] sps_id 1 out of range
[h264 @ 0x41d13dc0] sps_id 1 out of range


Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_63.mp4 | Total extracted frames: 2
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_89.mp4 | Total extracted frames: 7
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_65.mp4 | Total extracted frames: 17
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_94.mp4 | Total extracted frames: 13
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_78.mp4 | Total extracted frames: 9
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_85.mp4 | Total extracted frames: 26
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_75.mp4 | Total extracted frames: 14
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_36.mp4 | Total extracted frames: 4
Video: /home/ninhdao/D23_Lop4_Nhom16/data_collection/running/dog_running_23.mp4 | Total extracted frames: 8
Video: /home/ninhdao/D23

In [5]:
write_hog_csv(all_features, "/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/new_hog.csv")

In [6]:
import pandas as pd 

df = pd.DataFrame(all_features) 

df.head() 

,filename,filepath,label,keyframe_idx,frame_idx,timestamp_sec,fps,frame_interval,feature_dim,vector
0,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,0,0,0.000000,29.936401,30,8100,"[0.0, 0.0, 0.025137953, 0.02471043, 0.49198434..."
1,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,1,30,1.002124,29.936401,30,8100,"[0.064665966, 0.0, 0.010161277, 0.060379453, 0..."
2,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,2,60,2.004249,29.936401,30,8100,"[0.046225138, 0.0, 0.048020974, 0.049758114, 0..."
3,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,3,90,3.006373,29.936401,30,8100,"[0.12181008, 0.0, 0.020880662, 0.063198194, 0...."
4,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,4,120,4.008498,29.936401,30,8100,"[0.08931413, 0.0, 0.05025798, 0.016642706, 0.4..."


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5923 entries, 0 to 5922
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   filename        5923 non-null   object 
 1   filepath        5923 non-null   object 
 2   label           5923 non-null   object 
 3   keyframe_idx    5923 non-null   int64  
 4   frame_idx       5923 non-null   int64  
 5   timestamp_sec   5923 non-null   float64
 6   fps             5923 non-null   float64
 7   frame_interval  5923 non-null   int64  
 8   feature_dim     5923 non-null   int64  
 9   vector          5923 non-null   object 
dtypes: float64(2), int64(4), object(4)
memory usage: 462.9+ KB


In [8]:
df.to_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/new_hog.csv", index=False) 

In [1]:
from d23_lop4_nhom16.feature_extraction.color_histogram import extract_video_hsv_histogram 

In [2]:
kf = extract_video_hsv_histogram("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/video_test/77623-562965796.mp4")

In [3]:
kf

[{'video_path': '/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/video_test/77623-562965796.mp4',
  'frame_idx': 0,
  'timestamp': 0.0,
  'feature': array([3.07961998e-02, 2.70582562e-02, 2.31481481e-04, 9.09047068e-04,
         2.70061728e-05, 0.00000000e+00, 5.16010802e-05, 0.00000000e+00,
         0.00000000e+00, 7.93706597e-02, 2.05883488e-02, 8.68055556e-06,
         3.57007137e-02, 3.22415123e-02, 0.00000000e+00, 1.41878858e-03,
         1.44097222e-03, 0.00000000e+00, 5.12215471e-02, 1.50072338e-02,
         0.00000000e+00, 4.26117380e-01, 4.99165702e-02, 0.00000000e+00,
         3.45365548e-02, 1.67631173e-03, 0.00000000e+00, 7.91859568e-04,
         3.18287037e-05, 0.00000000e+00, 9.45216049e-05, 0.00000000e+00,
         0.00000000e+00, 5.78703704e-06, 0.00000000e+00, 0.00000000e+00,
         2.48408565e-03, 9.98263889e-05, 0.00000000e+00, 1.72646605e-04,
         0.00000000e+00, 0.00000000e+00, 3.85802469e-06, 0.00000000e+00,
         0.00000000e+00, 4.11458333e-03, 1.33439

In [6]:
import pandas as pd 

df = pd.read_csv("/home/ninhdao/D23_Lop4_Nhom16/src/d23_lop4_nhom16/features/new_hog.csv", skipinitialspace=True) 

df.head() 

,filename,filepath,label,keyframe_idx,frame_idx,timestamp_sec,fps,frame_interval,feature_dim,vector
0,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,0,0,0.000000,29.936401,30,8100,"[0.0,0.0,0.025137953460216522,0.02471042983233..."
1,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,1,30,1.002124,29.936401,30,8100,"[0.06466596573591232,0.0,0.010161276906728745,..."
2,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,2,60,2.004249,29.936401,30,8100,"[0.04622513800859451,0.0,0.048020973801612854,..."
3,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,3,90,3.006373,29.936401,30,8100,"[0.12181007862091064,0.0,0.02088066190481186,0..."
4,dog_eat_1.mp4,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Eating,4,120,4.008498,29.936401,30,8100,"[0.08931413292884827,0.0,0.050257980823516846,..."


In [1]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
from d23_lop4_nhom16.config import get_db_url
from d23_lop4_nhom16.database_loader import load_database_features
from d23_lop4_nhom16.extract_query_features import extract_query_features
from d23_lop4_nhom16.feature_scores import feature_scores
from d23_lop4_nhom16.fusions import normalize_and_ranking
from d23_lop4_nhom16.similarity.glcm_similarity import glcm_scaler 

In [2]:
query_video_path = Path(input()) 
print(f"\n=======================================================")
print(f"🔍 BẮT ĐẦU TÌM KIẾM VIDEO TƯƠNG ĐỒNG CHO: {query_video_path.name}")
print(f"=======================================================")
# 1. Kết nối DB
engine = create_engine(get_db_url())
# 2. Khối 1: Trích xuất đặc trưng video Query
q_feat = extract_query_features(query_video_path)
# 3. Khối 2: Tải đặc trưng toàn bộ DB vào RAM
df_videos, db_store = load_database_features(engine)
# 4. Fit StandardScaler cho GLCM trực tiếp từ DB (không dùng file pkl)
scaler_glcm = glcm_scaler(engine)
# 5. Khối 3: Tính điểm thô với từng video trong DB qua feature_scores()
print(
    f"⚡ Đang tính toán điểm tương đồng với {len(db_store)} video trong DB..."
)
raw_results = []
for vid, c_feat in db_store.items():
    # Gọi hàm trong feature_scores.py bạn đã viết
    scores = feature_scores(q_feat, c_feat, scaler_glcm=scaler_glcm)
    scores["video_id"] = vid
    raw_results.append(scores)
df_raw = pd.DataFrame(raw_results)
# 6. Khối 4, 5 & 6: Chuẩn hóa, Gộp điểm và Xếp hạng qua ranker.py
print(f"📊 Đang chuẩn hóa, gộp trọng số và xếp hạng...")
top_results = normalize_and_ranking(
    raw_scores=df_raw, df_videos=df_videos, top_k=5 
)
top_results 


🔍 BẮT ĐẦU TÌM KIẾM VIDEO TƯƠNG ĐỒNG CHO: test1.mp4
Stating extract...
Done fit
⚡ Đang tính toán điểm tương đồng với 501 video trong DB...
📊 Đang chuẩn hóa, gộp trọng số và xếp hạng...


,video_id,filepath,category,final_score,norm_hsv,norm_glcm,norm_hog,norm_flow,norm_mhi
0,287,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,running,0.813400,0.709265,0.157559,0.889132,0.959664,0.911707
1,135,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,walking,0.812187,0.573787,0.234159,1.000000,0.993193,0.848017
2,482,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,Stand,0.796811,0.666463,0.421470,0.898592,0.730111,0.974828
3,305,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,running,0.795864,0.605487,0.280482,0.891777,0.909039,0.904592
4,336,/home/ninhdao/D23_Lop4_Nhom16/data_collection/...,running,0.788558,0.582881,0.797652,0.775290,0.851351,0.844885


In [6]:
top_results['filepath'].iloc[2] 

'/home/ninhdao/D23_Lop4_Nhom16/data_collection/sit_stand/Stand/dog_standing_26.mp4'